# Sentiment Analysis of Twitter Data using Decision Tree Classifier

In [8]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv(r'C:\Users\Jerry\Desktop\Projects\Twitter_sentiment_analysis\data\Twitter_Data.csv')

In [14]:
df.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [15]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27480 entries, 0 to 27480
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   textID         27480 non-null  object
 1   text           27480 non-null  object
 2   selected_text  27480 non-null  object
 3   sentiment      27480 non-null  object
dtypes: object(4)
memory usage: 1.0+ MB


In [17]:
sample_df = df.sample(2000, random_state=42)

In [18]:
sample_df.head()

,textID,text,selected_text,sentiment
1589,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive
10414,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative
6562,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral
2603,984d753104,congrats hey,congrats,positive
4004,8a79072ca2,is texting,is texting,neutral


In [19]:
sample_df["sentiment"].value_counts()

sentiment
neutral     816
positive    641
negative    543
Name: count, dtype: int64

In [20]:
sample_df.reset_index(drop=True, inplace=True)

In [21]:
sample_df.head()

,textID,text,selected_text,sentiment
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral
3,984d753104,congrats hey,congrats,positive
4,8a79072ca2,is texting,is texting,neutral


In [22]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from nltk.tokenize import word_tokenize

In [23]:
sample_df_copy = sample_df.copy()

In [24]:
stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [25]:
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [26]:
stop_words.remove('not')
stop_words.remove('no')
stop_words.remove('nor')

In [27]:
stop_words

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'only',
 'or',
 'other',
 

In [28]:
def cleaning_data(data):
    # Remove URLs
    data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)
    # Remove user @ references and '#' from tweet
    data = re.sub(r'\@\w+|\#', '', data)
    # Remove punctuations
    data = re.sub(r'[^\w\s]', '', data)
    # Convert to lowercase
    data = data.lower()
    
    return data

In [29]:
sample_df["cleaned_text"] = sample_df["text"].apply(cleaning_data)

In [30]:
sample_df.head()

,textID,text,selected_text,sentiment,cleaned_text
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive,enjoy family trumps everything
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative,of them kinda turns me off of it all and the...
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral,clive its my birthday pat me
3,984d753104,congrats hey,congrats,positive,congrats hey
4,8a79072ca2,is texting,is texting,neutral,is texting


In [31]:
def preprocess_text(text):
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    cleaned_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    
    return ' '.join(cleaned_tokens)

In [32]:
sample_df["cleaned_text"] = sample_df["cleaned_text"].apply(preprocess_text)

In [33]:
sample_df.head()

,textID,text,selected_text,sentiment,cleaned_text
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive,enjoy family trump everything
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative,kinda turn buy dig deeper hole etc
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral,clive birthday pat
3,984d753104,congrats hey,congrats,positive,congrats hey
4,8a79072ca2,is texting,is texting,neutral,texting


In [34]:
def mapping_sentiment(sentiment):
    if sentiment == 'positive':
        return 2
    elif sentiment == 'neutral':
        return 1
    else:
        return 0

In [35]:
sample_df["mapped_sentiment"] = sample_df["sentiment"].apply(mapping_sentiment)

In [37]:
sample_df.head()

,textID,text,selected_text,sentiment,cleaned_text,mapped_sentiment
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive,enjoy family trump everything,2
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative,kinda turn buy dig deeper hole etc,0
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral,clive birthday pat,1
3,984d753104,congrats hey,congrats,positive,congrats hey,2
4,8a79072ca2,is texting,is texting,neutral,texting,1


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [39]:
tfidf_model = TfidfVectorizer(max_features=5000)

In [40]:
vectors = tfidf_model.fit_transform(sample_df["cleaned_text"]).toarray()

In [41]:
vectors

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(2000, 4642))

In [42]:
X = vectors
y = sample_df["mapped_sentiment"].values

In [43]:
y

array([2, 0, 1, ..., 1, 2, 1], shape=(2000,))

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [46]:
y_pred = dt_model.predict(X_test)

In [47]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [48]:
accuracy_score(y_test, y_pred)


0.5725

In [49]:
import pickle

with open('../models/sentiment_dt_model.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_model, f)

# Sentiment Analysis of Twitter Data using LSTM deep learning model

In [50]:
sample_df_copy.head()

,textID,text,selected_text,sentiment
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral
3,984d753104,congrats hey,congrats,positive
4,8a79072ca2,is texting,is texting,neutral


In [51]:
df["text"].iloc[10]

' as much as i love to be hopeful, i reckon the chances are minimal =P i`m never gonna get my cake and stuff'

In [52]:
import contractions
from spellchecker import SpellChecker


spell = SpellChecker()

def cleaning_data_(data):
    # Remove URLs
    data = re.sub(r'http\S+|www\S+|https\S+', '', data, flags=re.MULTILINE)
    
    # Remove user @ references and '#' from tweet
    data = re.sub(r'\@\w+|\#', '', data)

    data = contractions.fix(data)

    # Remove punctuations
    data = re.sub(r'[^\w\s]', '', data)

    words = word_tokenize(data)

    words = [spell.correction(word) for word in words if spell.correction(word)]

    data = ' '.join(words)

    # Convert to lowercase
    data = data.lower()
    
    return data

In [53]:
sample_df_copy.head()

,textID,text,selected_text,sentiment
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral
3,984d753104,congrats hey,congrats,positive
4,8a79072ca2,is texting,is texting,neutral


In [54]:
sample_df_copy["cleaned_text"] = sample_df_copy["text"].apply(cleaning_data_)

In [55]:
sample_df_copy.head()

,textID,text,selected_text,sentiment,cleaned_text
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive,enjoy family trumps everything
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative,of them kind of turns me off of it all and the...
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral,clive its my birthday pat me
3,984d753104,congrats hey,congrats,positive,congrats hey
4,8a79072ca2,is texting,is texting,neutral,is testing


In [56]:
sample_df_copy["mapped_sentiment"] = sample_df_copy["sentiment"].apply(mapping_sentiment)

In [57]:
sample_df_copy.head()

,textID,text,selected_text,sentiment,cleaned_text,mapped_sentiment
0,6c5505a37c,Enjoy! Family trumps everything,Enjoy! Family trumps everything,positive,enjoy family trumps everything,2
1,126b1e6a22,--of them kinda turns me off of it all. And ...,kinda turns me off,negative,of them kind of turns me off of it all and the...,0
2,5bc4e623c4,Clive it`s my birthday pat me http://apps.fac...,Clive it`s my birthday pat me,neutral,clive its my birthday pat me,1
3,984d753104,congrats hey,congrats,positive,congrats hey,2
4,8a79072ca2,is texting,is texting,neutral,is testing,1


In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

#import padsequences, pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

#import embedding layer
from tensorflow.keras.layers import Embedding

#import tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

#import lstm
from tensorflow.keras.layers import LSTM

In [59]:
X = sample_df_copy["cleaned_text"].values
y = sample_df_copy["mapped_sentiment"].values

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [60]:
max_words_vocab = 5000
sentence_length = 100
embedding_dim = 100
tokenizer = Tokenizer(num_words=max_words_vocab)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=sentence_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=sentence_length, padding='post', truncating='post')

In [63]:
X_train_pad.shape, X_test_pad.shape

((1600, 100), (400, 100))

In [64]:
y_train

array([0, 1, 0, ..., 0, 2, 0], shape=(1600,))

In [65]:
model = Sequential()
model.add(Embedding(input_dim=max_words_vocab, output_dim=embedding_dim, input_length=sentence_length))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(Dropout(0.5))
model.add(LSTM(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

c:\Users\Jerry\Desktop\Projects\Twitter_sentiment_analysis\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [66]:
history = model.fit(X_train_pad, y_train, epochs=10, batch_size=32, validation_data=(X_test_pad, y_test), verbose=1)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 15s 156ms/step - accuracy: 0.3969 - loss: 1.0903 - val_accuracy: 0.4125 - val_loss: 1.0809
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 180ms/step - accuracy: 0.4019 - loss: 1.0883 - val_accuracy: 0.4125 - val_loss: 1.0865
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 181ms/step - accuracy: 0.4069 - loss: 1.0888 - val_accuracy: 0.4125 - val_loss: 1.0807
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 196ms/step - accuracy: 0.4069 - loss: 1.0882 - val_accuracy: 0.4125 - val_loss: 1.0825
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 11s 212ms/step - accuracy: 0.4069 - loss: 1.0881 - val_accuracy: 0.4125 - val_loss: 1.0818
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 190ms/step - accuracy: 0.4069 - loss: 1.0862 - val_accuracy: 0.4125 - val_loss: 1.0785
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 190ms/step - accuracy: 0.4069 - loss: 1.0881 - val_accuracy: 0.4125 - val_loss: 1.0834
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 183ms/step - accuracy: 0.4069 - loss: 1.0886 - val_accuracy

In [67]:
y_pred = model.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 87ms/step


In [68]:
y_pred_classes

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [69]:
model.save('../models/sentiment_lstm_model.keras')

In [70]:
from pickle import dump
dump(tokenizer, open('../models/tokenizer.pkl', 'wb'))